# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup

In [2]:
import sys
import os

# Project root (absolute path)
PROJECT_ROOT = os.path.expanduser('~/claudework/llm-ap-generation/llm-ap-generation')
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

from pathlib import Path
from cve2pddlap.evaluation import (
    create_ff_checker, create_enhsp_checker, PlannerResult
)
from cve2pddlap.core.data_loader import load_few_shot_pool

ff = create_ff_checker()
enhsp = create_enhsp_checker()

print(f'Project root: {PROJECT_ROOT}')
print(f'Metric-FF: {ff.ff_path}')
print(f'ENHSP:     {enhsp.jar_path}')
print('Evaluation tools loaded successfully')

Project root: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation
Metric-FF: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/tools/Metric-FF/ff
ENHSP:     /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/tools/enhsp/enhsp.jar
Evaluation tools loaded successfully


## 2. Data

In [3]:
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
EXPERIMENTS_DIR = os.path.join(PROJECT_ROOT, 'experiments')

# Load all reference examples (CVE / AP pairs)
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
print(f'Experiments dir: {EXPERIMENTS_DIR}')
print(f'Experiments exist: {os.path.isdir(EXPERIMENTS_DIR)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')

Reference examples: 55
Experiments dir: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/experiments
Experiments exist: True
  CVE-2022-1471 / AP1
  CVE-2022-40149 / AP1
  CVE-2022-40149 / AP2
  CVE-2022-40150 / AP1
  CVE-2022-40150 / AP2
  ...


## 3. Solvability and Syntax Check for Totally 55 Reference Domain and Problem
Validate generated PDDL with Metric-FF (plan finding) and ENHSP (strict syntax).

In [ ]:
# Test solvability on all reference domain/problem pairs
import os

results = []
for ex in few_shot_pool:
    cve_dir = os.path.join(DATASET_PATH, ex.cve_id, ex.ap_id)
    domain_path = os.path.join(cve_dir, 'domain.pddl')
    problem_path = os.path.join(cve_dir, 'problem.pddl')

    if not os.path.exists(problem_path):
        print(f'  SKIP {ex.key}: no problem.pddl')
        continue

    r_ff = ff.check(domain_path, problem_path)
    r_enhsp = enhsp.check(domain_path, problem_path)

    results.append({
        'key': ex.key,
        'ff_solvable': r_ff.solvable,
        'ff_plan_length': r_ff.plan_length,
        'ff_cost': r_ff.plan_cost,
        'enhsp_syntax_ok': r_enhsp.success,
        'enhsp_error': r_enhsp.error,
    })

    status = '✓' if r_ff.solvable else '✗'
    syntax = '✓' if r_enhsp.success else '✗'
    print(f'  {ex.key:35s}  FF:{status} (len={r_ff.plan_length}, cost={r_ff.plan_cost})  ENHSP:{syntax}')

print(f'\nTotal: {len(results)} | '
      f'FF solvable: {sum(1 for r in results if r["ff_solvable"])} | '
      f'ENHSP syntax OK: {sum(1 for r in results if r["enhsp_syntax_ok"])}')

  CVE-2022-1471 / AP1                  FF:✓ (len=13, cost=15.0)  ENHSP:✓
  CVE-2022-40149 / AP1                 FF:✓ (len=8, cost=37.0)  ENHSP:✓
  CVE-2022-40149 / AP2                 FF:✓ (len=9, cost=38.0)  ENHSP:✓
  CVE-2022-40150 / AP1                 FF:✓ (len=8, cost=37.0)  ENHSP:✓


## 4. Evaluate Domain PDDL (Generated + Reference)
Select domain.pddl files to evaluate. The problem.pddl is **auto-generated** from the domain using code (no LLM).
Then verify solvability (Metric-FF) and syntax (ENHSP).

**Sources:**
- `experiments/` — LLM-generated domains
- `resources/data/CVE-PDDL-NNL-ReAP/` — reference domains (55 APs)

In [18]:
import sys, os, re, glob
import ipywidgets as widgets
from IPython.display import display, clear_output

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem

DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
EXPERIMENTS_DIR = os.path.join(PROJECT_ROOT, 'experiments')

ff = create_ff_checker()
enhsp = create_enhsp_checker()

# --- Build domain list: generated + reference ---
def _scan_domains():
    domains = []
    for f in sorted(glob.glob(os.path.join(EXPERIMENTS_DIR, '*.pddl'))):
        if not f.endswith('_problem.pddl'):
            label = f'[generated] {os.path.basename(f)}'
            domains.append((label, f))
    for f in sorted(glob.glob(os.path.join(DATASET_PATH, '*/AP*/domain.pddl'))):
        parts = f.split('/')
        key = f'{parts[-3]}/{parts[-2]}'
        label = f'[reference] {key}'
        domains.append((label, f))
    return domains

domain_options = _scan_domains()

domain_select = widgets.SelectMultiple(
    options=domain_options,
    description='Domains:',
    rows=15,
    layout=widgets.Layout(width='600px'),
    style={'description_width': '70px'},
)

select_all_gen_btn = widgets.Button(description='Select all generated', layout=widgets.Layout(width='180px'))
select_all_ref_btn = widgets.Button(description='Select all reference', layout=widgets.Layout(width='180px'))
select_all_btn = widgets.Button(description='Select all', layout=widgets.Layout(width='120px'))
clear_btn = widgets.Button(description='Clear', layout=widgets.Layout(width='80px'))
refresh_btn = widgets.Button(description='Refresh list', button_style='warning', layout=widgets.Layout(width='120px'))

def _select_all_gen(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[generated]')]
def _select_all_ref(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[reference]')]
def _select_all(b):
    domain_select.value = [v for _, v in domain_options]
def _clear(b):
    domain_select.value = []
def _refresh(b):
    global domain_options
    domain_options = _scan_domains()
    domain_select.options = domain_options
    print(f'Refreshed: {len(domain_options)} domains found')

select_all_gen_btn.on_click(_select_all_gen)
select_all_ref_btn.on_click(_select_all_ref)
select_all_btn.on_click(_select_all)
clear_btn.on_click(_clear)
refresh_btn.on_click(_refresh)

display(widgets.VBox([
    widgets.HTML('<b>Select domains to evaluate</b> (Ctrl/Cmd+click for multi-select)'),
    widgets.HBox([select_all_gen_btn, select_all_ref_btn, select_all_btn, clear_btn, refresh_btn]),
    domain_select,
]))
print(f'Total available: {len(domain_options)} domains')

Total available: 63 domains


In [19]:
# Step 1: Generate problem.pddl for each selected domain (code-based, no LLM)

selected_domains = list(domain_select.value)
if not selected_domains:
    raise ValueError('No domains selected. Run the cell above and select domains first.')

print(f'Generating problem.pddl for {len(selected_domains)} domain(s)...\n')

eval_items = []  # (display_name, source, domain_path, problem_pddl)

for domain_path in selected_domains:
    if EXPERIMENTS_DIR in domain_path:
        source = 'generated'
        display_name = os.path.basename(domain_path).replace('.pddl', '')
    else:
        parts = domain_path.split('/')
        source = 'reference'
        display_name = f'{parts[-3]}/{parts[-2]}'

    try:
        with open(domain_path) as f:
            domain_pddl = f.read()
        problem_pddl = generate_problem(domain_pddl)
        eval_items.append((display_name, source, domain_path, problem_pddl))
        print(f'  \u2713 {display_name} [{source}]')
    except Exception as e:
        print(f'  \u2717 {display_name} [{source}] — ERROR: {e}')

print(f'\nProblem generation done: {len(eval_items)}/{len(selected_domains)} succeeded')

Generating problem.pddl for 1 domain(s)...

  ✓ CVE-2025-66032_qwen-max_1shot [generated]

Problem generation done: 1/1 succeeded


In [15]:
# Step 2 (optional): Preview generated problem.pddl
# Change idx to inspect different domains

idx = 0  # change this

if eval_items:
    name, source, dpath, problem = eval_items[idx]
    print(f'=== {name} [{source}] ===')
    print(f'Domain: {dpath}\n')
    print(problem)
else:
    print('No items. Run the cell above first.')

=== CVE-2025-66032_qwen-max_1shot [generated] ===
Domain: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/experiments/CVE-2025-66032_qwen-max_1shot.pddl

(define (problem AEDI-elevation-of-privilege)
  (:domain AED)
  (:objects
    arbitrary-code_SEFA - arbitrary-code
    coding-tool_SEFA - coding-tool
    context-window_SEFA - context-window
    SEFA - target-system
    untrusted-content_SEFA - untrusted-content
    validation-bypass_SEFA - validation-bypass
  )
  (:init
    (= (total-cost) 0)
    (has-coding-tool SEFA coding-tool_SEFA)
    (context-window-exists SEFA context-window_SEFA)
    (shell-command-parsing-error coding-tool_SEFA)
    (ifs-variable-exploitable coding-tool_SEFA)
    (short-cli-flag-exploitable coding-tool_SEFA)
    (= (version coding-tool_SEFA) 100092000)
  )
  (:goal (and (elevation-of-privilege SEFA)))
  (:metric minimize (total-cost)))



In [20]:
# Step 3: Evaluate solvability (Metric-FF) and syntax (ENHSP)

import tempfile

eval_results = []
print(f'Running FF + ENHSP on {len(eval_items)} domain(s)...\n')

for display_name, source, domain_path, problem_pddl in eval_items:
    # Write problem to temp file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.pddl', delete=False) as tmp:
        tmp.write(problem_pddl)
        tmp_path = tmp.name

    r_ff = ff.check(domain_path, tmp_path)
    r_enhsp = enhsp.check(domain_path, tmp_path)
    os.unlink(tmp_path)

    ff_ok = '\u2713' if r_ff.solvable else '\u2717'
    enhsp_ok = '\u2713' if r_enhsp.success else '\u2717'

    eval_results.append({
        'name': display_name,
        'source': source,
        'ff_solvable': r_ff.solvable,
        'ff_plan_length': r_ff.plan_length,
        'ff_cost': r_ff.plan_cost,
        'ff_error': r_ff.error,
        'ff_plan': r_ff.plan,
        'enhsp_ok': r_enhsp.success,
        'enhsp_error': r_enhsp.error,
    })

    print(f'  {display_name} [{source}]')
    print(f'    FF: {ff_ok}  len={r_ff.plan_length}  cost={r_ff.plan_cost}')
    if r_ff.error:
        print(f'    FF error: {r_ff.error[:200]}')
    print(f'    ENHSP: {enhsp_ok}')
    if r_enhsp.error:
        print(f'    ENHSP error: {r_enhsp.error[:200]}')
    if r_ff.plan:
        print(f'    Plan ({len(r_ff.plan)} steps):')
        for i, a in enumerate(r_ff.plan):
            print(f'      {i}: {a}')
    print()

# Summary
n = len(eval_results)
n_ff = sum(1 for r in eval_results if r['ff_solvable'])
n_enhsp = sum(1 for r in eval_results if r['enhsp_ok'])
print(f'=== Summary ===')
print(f'Total: {n} | FF solvable: {n_ff}/{n} | ENHSP syntax OK: {n_enhsp}/{n}')

Running FF + ENHSP on 1 domain(s)...

  CVE-2025-66032_qwen-max_1shot [generated]
    FF: ✓  len=4  cost=33.0
    ENHSP: ✓
    Plan (4 steps):
      0: ATTACKER-INJECTS-UNTRUSTED-CONTENT-INTO-CONTEXT-WINDOW SEFA UNTRUSTED-CONTENT_SEFA CONTEXT-WINDOW_SEFA
      1: ATTACKER-MANIPULATES-CLI-FLAGS SEFA UNTRUSTED-CONTENT_SEFA CLI-FLAG_SEFA
      2: ATTACKER-INJECTS-SHELL-COMMAND SEFA CLI-FLAG_SEFA SHELL-COMMAND_SEFA
      3: TARGET-SYSTEM-EXECUTES-ARBITRARY-CODE SEFA SHELL-COMMAND_SEFA

=== Summary ===
Total: 1 | FF solvable: 1/1 | ENHSP syntax OK: 1/1


In [21]:
import sys, os, re, glob
import ipywidgets as widgets
from IPython.display import display, clear_output

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem

DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
EXPERIMENTS_DIR = os.path.join(PROJECT_ROOT, 'experiments')

ff = create_ff_checker()
enhsp = create_enhsp_checker()

# --- Build domain list: generated + reference ---
def _scan_domains():
    domains = []
    for f in sorted(glob.glob(os.path.join(EXPERIMENTS_DIR, '*.pddl'))):
        if not f.endswith('_problem.pddl'):
            label = f'[generated] {os.path.basename(f)}'
            domains.append((label, f))
    for f in sorted(glob.glob(os.path.join(DATASET_PATH, '*/AP*/domain.pddl'))):
        parts = f.split('/')
        key = f'{parts[-3]}/{parts[-2]}'
        label = f'[reference] {key}'
        domains.append((label, f))
    return domains

domain_options = _scan_domains()

domain_select = widgets.SelectMultiple(
    options=domain_options,
    description='Domains:',
    rows=15,
    layout=widgets.Layout(width='600px'),
    style={'description_width': '70px'},
)

select_all_gen_btn = widgets.Button(description='Select all generated', layout=widgets.Layout(width='180px'))
select_all_ref_btn = widgets.Button(description='Select all reference', layout=widgets.Layout(width='180px'))
select_all_btn = widgets.Button(description='Select all', layout=widgets.Layout(width='120px'))
clear_btn = widgets.Button(description='Clear', layout=widgets.Layout(width='80px'))
refresh_btn = widgets.Button(description='Refresh list', button_style='warning', layout=widgets.Layout(width='120px'))

def _select_all_gen(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[generated]')]
def _select_all_ref(b):
    domain_select.value = [v for l, v in domain_options if l.startswith('[reference]')]
def _select_all(b):
    domain_select.value = [v for _, v in domain_options]
def _clear(b):
    domain_select.value = []
def _refresh(b):
    global domain_options
    domain_options = _scan_domains()
    domain_select.options = domain_options
    print(f'Refreshed: {len(domain_options)} domains found')

select_all_gen_btn.on_click(_select_all_gen)
select_all_ref_btn.on_click(_select_all_ref)
select_all_btn.on_click(_select_all)
clear_btn.on_click(_clear)
refresh_btn.on_click(_refresh)

display(widgets.VBox([
    widgets.HTML('<b>Select domains to evaluate</b> (Ctrl/Cmd+click for multi-select)'),
    widgets.HBox([select_all_gen_btn, select_all_ref_btn, select_all_btn, clear_btn, refresh_btn]),
    domain_select,
]))
print(f'Total available: {len(domain_options)} domains')

Total available: 63 domains


In [30]:
# Step 1: Generate problem.pddl for each selected domain (code-based, no LLM)

selected_domains = list(domain_select.value)
if not selected_domains:
    raise ValueError('No domains selected. Run the cell above and select domains first.')

print(f'Generating problem.pddl for {len(selected_domains)} domain(s)...\n')

eval_items = []  # (display_name, source, domain_path, problem_pddl)

for domain_path in selected_domains:
    if EXPERIMENTS_DIR in domain_path:
        source = 'generated'
        display_name = os.path.basename(domain_path).replace('.pddl', '')
    else:
        parts = domain_path.split('/')
        source = 'reference'
        display_name = f'{parts[-3]}/{parts[-2]}'

    try:
        with open(domain_path) as f:
            domain_pddl = f.read()
        problem_pddl = generate_problem(domain_pddl)
        eval_items.append((display_name, source, domain_path, problem_pddl))
        print(f'  \u2713 {display_name} [{source}]')
    except Exception as e:
        print(f'  \u2717 {display_name} [{source}] — ERROR: {e}')

print(f'\nProblem generation done: {len(eval_items)}/{len(selected_domains)} succeeded')

Generating problem.pddl for 1 domain(s)...

  ✓ CVE-2025-66032_gpt4_1shot [generated]

Problem generation done: 1/1 succeeded


In [31]:
# Step 2 (optional): Preview generated problem.pddl
# Change idx to inspect different domains

idx = 0  # change this

if eval_items:
    name, source, dpath, problem = eval_items[idx]
    print(f'=== {name} [{source}] ===')
    print(f'Domain: {dpath}\n')
    print(problem)
else:
    print('No items. Run the cell above first.')

=== CVE-2025-66032_gpt4_1shot [generated] ===
Domain: /Users/cuilin/claudework/llm-ap-generation/llm-ap-generation/experiments/CVE-2025-66032_gpt4_1shot.pddl

(define (problem AEDI-elevation-of-privilege)
  (:domain AED)
  (:objects
    coding-tool_SEFA - coding-tool
    context-window_SEFA - context-window
    SEFA - target-system
    untrusted-content_SEFA - untrusted-content
  )
  (:init
    (= (total-cost) 0)
    (has-coding-tool SEFA coding-tool_SEFA)
    (vulnerable-claude-code coding-tool_SEFA)
    (context-window-accepts-untrusted-content coding-tool_SEFA context-window_SEFA)
    (= (version coding-tool_SEFA) 1009199000)
  )
  (:goal (and (elevation-of-privilege SEFA)))
  (:metric minimize (total-cost)))



In [32]:
# Step 3: Evaluate solvability (Metric-FF) and syntax (ENHSP)

import tempfile

eval_results = []
print(f'Running FF + ENHSP on {len(eval_items)} domain(s)...\n')

for display_name, source, domain_path, problem_pddl in eval_items:
    # Write problem to temp file
    with tempfile.NamedTemporaryFile(mode='w', suffix='.pddl', delete=False) as tmp:
        tmp.write(problem_pddl)
        tmp_path = tmp.name

    r_ff = ff.check(domain_path, tmp_path)
    r_enhsp = enhsp.check(domain_path, tmp_path)
    os.unlink(tmp_path)

    ff_ok = '\u2713' if r_ff.solvable else '\u2717'
    enhsp_ok = '\u2713' if r_enhsp.success else '\u2717'

    eval_results.append({
        'name': display_name,
        'source': source,
        'ff_solvable': r_ff.solvable,
        'ff_plan_length': r_ff.plan_length,
        'ff_cost': r_ff.plan_cost,
        'ff_error': r_ff.error,
        'ff_plan': r_ff.plan,
        'enhsp_ok': r_enhsp.success,
        'enhsp_error': r_enhsp.error,
    })

    print(f'  {display_name} [{source}]')
    print(f'    FF: {ff_ok}  len={r_ff.plan_length}  cost={r_ff.plan_cost}')
    if r_ff.error:
        print(f'    FF error: {r_ff.error[:200]}')
    print(f'    ENHSP: {enhsp_ok}')
    if r_enhsp.error:
        print(f'    ENHSP error: {r_enhsp.error[:200]}')
    if r_ff.plan:
        print(f'    Plan ({len(r_ff.plan)} steps):')
        for i, a in enumerate(r_ff.plan):
            print(f'      {i}: {a}')
    print()

# Summary
n = len(eval_results)
n_ff = sum(1 for r in eval_results if r['ff_solvable'])
n_enhsp = sum(1 for r in eval_results if r['enhsp_ok'])
print(f'=== Summary ===')
print(f'Total: {n} | FF solvable: {n_ff}/{n} | ENHSP syntax OK: {n_enhsp}/{n}')

Running FF + ENHSP on 1 domain(s)...

  CVE-2025-66032_gpt4_1shot [generated]
    FF: ✓  len=3  cost=32.0
    ENHSP: ✓
    Plan (3 steps):
      0: ATTACKER-INJECTS-UNTRUSTED-CONTENT-INTO-CONTEXT-WINDOW SEFA CONTEXT-WINDOW_SEFA UNTRUSTED-CONTENT_SEFA CODING-TOOL_SEFA
      1: TARGET-SYSTEM-TRIGGERS-ARBITRARY-CODE-EXECUTION SEFA CONTEXT-WINDOW_SEFA UNTRUSTED-CONTENT_SEFA
      2: TARGET-SYSTEM-EXPERIENCES-CODE-EXECUTION-VULNERABILITY-IMPACT SEFA

=== Summary ===
Total: 1 | FF solvable: 1/1 | ENHSP syntax OK: 1/1


## Syntactic Metrics (future)

Synonym normalization, variable name normalization, classical metrics (TP/FP/FN/Precision/Recall/F1).

In [ ]:
# TODO: syntactic evaluation

## Semantic Metrics (future)

### Embedding Similarity
Cosine similarity between NL descriptions and generated PDDL (ref: Planning in the Dark).

In [ ]:
# TODO: embedding-based evaluation

### LLM-based Evaluation

#### Intrinsic
CVE description + generated PDDL → LLM judges quality (feasibility, atomicity, completeness).

Specification-Code embedding similarity

In [ ]:
# TODO: LLM-based intrinsic evaluation

#### Extrinsic
CVE description + reference PDDL + generated PDDL → LLM judges alignment.

In [ ]:
# TODO: LLM-based extrinsic evaluation

### Human Evaluation

5-point Likert scale, ≥3 experts, Krippendorff's alpha. Structured questions at action-level and path-level.

In [ ]:
# TODO: human evaluation questionnaire design